# Edge ML Audio Compression Benchmark
Voxserv `mono_44100`, 20 ms frames (882 samples), split INT8 encoder/decoder.


In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent/'python'))
from models import build_autoencoder, build_encoder, build_decoder
from dataset import load_training_frames
frames=load_training_frames('../data/wav')
frames.shape

Found 9 WAV files in ../data/wav.


(26297, 882)

In [2]:
ae=build_autoencoder()
ae.summary()
ae.compile(optimizer='adam', loss='mse')
x=frames[...,None]
ae.fit(x,x,epochs=40,batch_size=32,validation_split=.1,shuffle=True)

C:\Users\Khalil\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "audio_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ audio (InputLayer)                   │ (None, 882, 1)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ audio_encoder (Functional)           │ (None, 128)                 │         233,776 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ audio_decoder (Functional)           │ (None, 882, 1)              │         235,409 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 469,185 (1.79 MB)

 Trainable params: 469,185 (1.79 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 8.7008e-04 - val_loss: 4.0186e-04
Epoch 2/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 4.0959e-04 - val_loss: 3.7281e-04
Epoch 3/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 3.5276e-04 - val_loss: 3.5880e-04
Epoch 4/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 3.2462e-04 - val_loss: 3.5865e-04
Epoch 5/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 3.1401e-04 - val_loss: 3.4846e-04
Epoch 6/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 3.0586e-04 - val_loss: 3.4369e-04
Epoch 7/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 2.9867e-04 - val_loss: 3.4365e-04
Epoch 8/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 2.9645e-04 - val_loss: 3.3906e-04
Epoch 9/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 2.9120e-04 - val_loss: 3.4240e-04
Epoch 10/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 2.8962e-04 - val_loss: 3.3722e-04
Epoch 11/40
740/740 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss

In [4]:
# (Cell 4 in your notebook)
ae.save('../models/audio_autoencoder.keras')
enc=build_encoder(); dec=build_decoder()
enc.set_weights(ae.get_layer('audio_encoder').get_weights())
dec.set_weights(ae.get_layer('audio_decoder').get_weights())
enc.save('../models/audio_encoder.keras'); dec.save('../models/audio_decoder.keras')

## Quantization
Run `python ../python/quantize.py --representative_dir ../data/wav`. The script performs full integer INT8 conversion for both models. Record the actual `.tflite` sizes and quantization parameters.

In [ ]:
# Shell: python ../python/quantize.py --representative_dir ../data/wav
# Shell: python ../python/inspect_tflite.py ../models/audio_encoder_int8.tflite ../models/audio_decoder_int8.tflite

## Compression accounting
PCM = 882*16 = 14112 bits/frame = 705.6 kbit/s.
Neural payload = 128 bytes/frame = 51.2 kbit/s before framing.
IMA ADPCM = 4 bits/sample = 176.4 kbit/s before state/header overhead.
Use measured reconstruction metrics and actual firmware telemetry in the final README.